# Rulebook — a RAG walkthrough

This notebook follows a question through the whole pipeline, one stage at a time, so you can
*see* what Retrieval-Augmented Generation actually does: **ingest → chunk → embed → retrieve →
generate**. It runs against the two committed sample domains (`checkers` and `cornhole`), so no
downloads are needed.

**Before you start:**
1. `uv sync` and copy `.env.example` to `.env` with your keys (`ANTHROPIC_API_KEY` + an embedding key).
2. Build the sample index: `uv run python scripts/build_index.py`
3. Launch Jupyter **from the repo root**: `uv run jupyter lab`

The first cell makes the working directory the repo root and loads `.env`, so the imports below find your keys.

In [ ]:
import os, pathlib
# Walk up to the repo root (where pyproject.toml lives) so .env and ./data/index resolve.
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root)
from dotenv import load_dotenv
load_dotenv(root / '.env')
print('working dir:', os.getcwd())
assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY in .env first'

## 1. Ingest + chunk — what is a "chunk"?

RAG retrieves *pieces* of documents, not whole files. The quality of those pieces is the single
biggest lever on the whole system. `ingest` pulls text (with page numbers); `chunking` splits it on
the document's own rule numbering, so each chunk is one self-contained rule you can cite — not a
500-character slice through the middle of a sentence.

In [ ]:
from rulebook.ingest import extract_pages
from rulebook.chunking import chunk_pages

doc = pathlib.Path('rules/checkers/checkers.md')
pages = extract_pages(doc)
chunks = chunk_pages(pages, source=doc.name, domain='checkers')

print(f'{len(chunks)} chunks from {doc.name}')
for c in chunks[:5]:
    print(f'  [{c.rule_id:<8}] {c.text[:70]}...')

Notice the `rule_id` on each chunk (`1.1`, `2.3`, …). That id is what the final answer cites, which
is what lets a reader verify a claim against the actual rule. Fixed-size chunking would throw it away.

## 2. Embed — turning text into vectors

An **embedding** maps text to a fixed-length vector so that *similar meanings land near each other*.
Retrieval is then just "which stored vectors are nearest the question's vector?". Let's embed four
short phrases and look at their cosine similarities — two about the same idea, two unrelated.

In [ ]:
import numpy as np
from rulebook.app_state import initialize
from rulebook.config import settings
from rulebook.embeddings import get_embedder

initialize(settings)  # constructs the cost-governor singletons the embed call records through
embedder = get_embedder()

phrases = [
    'a king may move backward',
    'kings can move in reverse',   # ~ same meaning as the first
    'a bag in the hole scores three points',
    'the corn bag went through the hole for 3',  # ~ same meaning as the third
]
vecs = np.array(embedder.embed(phrases, input_type='document'))
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
sim = vecs @ vecs.T  # cosine similarity (unit vectors)

import itertools
for i, j in itertools.combinations(range(len(phrases)), 2):
    print(f'{sim[i, j]:.3f}  |  {phrases[i][:34]:<34}  ~?~  {phrases[j][:34]}')

The two paraphrase pairs score far higher than the unrelated crossings — the embedder has no shared
words to go on ("king"/"reverse", "bag"/"corn"), only meaning. That is the whole trick retrieval rests on.

## 3. Retrieve — ask without the LLM yet

`retrieve` embeds the question and returns the nearest stored chunks. No generation yet — just the raw
passages and their distances (L2 on unit vectors: **lower = nearer**). This needs the index built (step 2
of the prerequisites).

In [ ]:
from rulebook.retrieve import retrieve

hits = retrieve('can a king move backwards?', domain='checkers', k=3)
for h in hits:
    print(f'  distance={h.distance:.3f}  [{h.domain} {h.rule_id}]  {h.text[:70]}...')

The nearest chunk should be checkers rule `4.2` ("How a king moves") — retrieval found the right rule
from a question that doesn't share its exact words. Try editing the question and re-running.

## 4. Generate — add the LLM, with citations forced

`ask` runs the whole pipeline: retrieve, then prompt Claude with *only* those chunks and a rule that it
must cite every claim as `[domain rule_id]` and must refuse anything not in the context. That refusal is
what separates RAG from a plausible-sounding guess.

In [ ]:
from rulebook.pipeline import ask

res = ask('How does a piece become a king, and what can it then do?', domain='checkers')
print(res.answer)
print('\nretrieved:', [f'{c.domain} {c.rule_id}' for c in res.chunks])
print('unverified citations:', res.unverified_citations)

### The guardrail: asking something that isn't in the rulebook

Now ask about a game the corpus doesn't contain. A naive LLM would happily invent an answer from its
training data; here the prompt forces an honest "not in the provided rules," and `unverified_citations`
would flag any rule id the model cited that wasn't actually retrieved.

In [ ]:
res = ask('What is the offside rule in soccer?', domain='checkers')
print(res.answer)
print('\nunverified citations:', res.unverified_citations)

## 5. Cross-domain — compare two rulesets

With more than one domain built, `ask` can retrieve from each and let the model compare. Here it draws on
both `checkers` and `cornhole` at once.

In [ ]:
res = ask('How do you win each game?', domains=['checkers', 'cornhole'])
print(res.answer)
print('\nretrieved from:', sorted({c.domain for c in res.chunks}))

## Where to go next

- **Turn a knob.** Re-run step 3 with `k=1` vs `k=8`, or rebuild after editing a chunk, and watch the
  answer change. RAG quality is mostly a tuning problem.
- **Add a real ruleset.** `uv run python -m rulebook.add <slug> <your.pdf>` ingests and indexes any rule
  doc you download — see `rules/SOURCES.md` for a menu of recognizable sports.
- **Read the six core modules** in order: `ingest` → `chunking` → `embeddings` → `store` → `retrieve` →
  `generate`. Everything else in `src/rulebook/` is production scaffolding (auth, roles, cost governance).